# RL 14 — RLHF and Preference Learning

Preference learning replaces hand-written rewards with human comparisons. This matters for robots and assistants when the real objective is hard to specify.

**Learning style:** mechanisms first → frameworks second → real systems third. The notebook is intentionally slow, explicit, and beginner-friendly.

In [ ]:
# Setup: run this first.
# Works from the repository root. In Colab, clone the repo first, then run from inside it.
from pathlib import Path
import sys, math, random
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    print('Tip: run this notebook from the repository root, or clone the repo in Colab first.')
sys.path.insert(0, str(ROOT))
print('Working directory:', ROOT)

## Notebook type and ordered study structure

**Notebook type:** Type A — core mechanism notebook. This should be studied slowly because the mechanism appears again and again across the whole repo.

**Your objective in this notebook:** Understand reward learning from preferences.

Use this exact order every time:

1. **Why this matters** — identify the real problem this topic solves.
2. **Mental model** — explain the idea in plain language before symbols.
3. **Math mechanism** — write the smallest formula and define every symbol.
4. **From-scratch code** — run/read the simple implementation slowly.
5. **Line-by-line explanation** — trace inputs, internal variables, update rule, and outputs.
6. **Debug/visualize** — print shapes, values, curves, maps, or weights so behavior is visible.
7. **Framework version** — map the mechanism to a real library/API.
8. **Scratch → framework mapping** — write what the framework hides and what it exposes.
9. **Real-system role** — place the topic inside robotics, driving, drones, manipulation, or VLA.
10. **Failure modes** — list how it breaks and how you would notice.
11. **Exercises** — change parameters, break the example, and explain the result.
12. **Mini-project** — build a small artifact you can keep.
13. **Next step** — choose the next notebook or tool.

**Mental model for this topic:** Humans can compare outcomes even when writing a reward is hard.

**Core math / mechanism to keep in mind:** P(A>B)=sigmoid(R(A)-R(B)).

**Recommended debug habit:** after every code cell, ask “what are the inputs, what changed, and what would be unsafe or wrong in a real robot?”

## From-scratch focus and code-reading checklist

**Scratch focus:** Tiny preference score over success/collision features.

When reading code in this notebook or the matching repo module, trace it like this:

| Step | Question to answer |
|---|---|
| Input | What is the state, observation, tensor, point, reward, or measurement? |
| Representation | Is it a scalar, vector, matrix, image, point cloud, token sequence, or action? |
| Mechanism | Which line implements the math/update rule? |
| Parameters | Which numbers are hyperparameters, physical constants, or learned weights? |
| Output | What changed after the step? |
| Debug signal | What should I print/plot to know it is working? |

Do not move to the framework/API version until you can explain the scratch version without reading the code comments.

## Visualization and debugging ideas

Use at least one of these while studying:

- Print tensor/vector shapes before and after the core operation.
- Print the first few values before and after an update.
- Plot a curve when there is learning, control, filtering, planning, or optimization.
- Draw frames, maps, paths, sensor rays, or attention matrices when geometry is involved.
- Change one parameter at a time and predict the effect before running.

For this notebook, a useful first visualization/debug target is: **Tiny preference score over success/collision features.**

## Scratch → framework mapping

| From-scratch idea in this repo | Practical framework/API | What to learn from the framework |
|---|---|---|
| linear reward model | TRL/reward models | preference learning scales to neural reward models |
| pairwise comparisons | DPO/RLHF datasets | frameworks package preference optimization |
| trajectory features | human-in-loop robotics | robot preferences compare behaviors |

**Framework learning rule:** do not memorize the API first. First identify which scratch concept it replaces, then learn its inputs, outputs, configuration, and failure modes.

## Real-system application

Robots may need preferences for smoothness, caution, social behavior, or task style.

Ask these system questions:

1. What module produces the input to this component?
2. What module consumes its output?
3. What latency, safety, calibration, or data-format assumptions exist?
4. What metric tells me this component is good enough for the larger system?

## Failure modes and debugging

Common ways this topic can fail:

- reward model hacking
- biased preferences
- missing safety constraints

For each failure, write:

- **Symptom:** what would I see in logs, plots, robot behavior, or evaluation?
- **Likely cause:** what assumption broke?
- **First debug action:** what is the smallest thing to inspect?

## Mini-project and mastery checklist

**Mini-project:** Collect five preference pairs for robot reaching trajectories and fit a linear score.

Mastery checklist:

- [ ] I can explain the mental model in one paragraph.
- [ ] I can write the core formula and define every symbol.
- [ ] I can run or read the scratch code and point to the core update/operation.
- [ ] I can name the production framework/API version of the same idea.
- [ ] I can describe where this topic sits in a robot/car/drone/VLA stack.
- [ ] I can name at least three failure modes and one debug action for each.

**Next study steps:** Part 12 imitation, RL 15 safe RL, Part 10 VLA

## 1. Mental model

Preference learning replaces hand-written rewards with human comparisons. This matters for robots and assistants when the real objective is hard to specify.

Before code, write one sentence in your own words: *what problem does this topic solve?*

## 2. Mechanism and math

A reward model can be trained from pairwise preferences:
\[
P(A \succ B)=\sigma(R_\phi(A)-R_\phi(B))
\]
Then a policy is optimized against the learned reward, often with a constraint to stay near the original policy.

## 3. From-scratch lab

Fit a tiny preference score by hand: preferred trajectories should get higher scores.

Read every line. The code avoids clever abstractions so you can see the mechanism.

In [ ]:
# Tiny linear reward model: score = w_success*success - w_collision*collisions
prefs = [({'success':1,'collisions':0}, {'success':0,'collisions':0}),
         ({'success':1,'collisions':0}, {'success':1,'collisions':2})]
w_success, w_collision = 1.0, 1.0
for A, B in prefs:
    score_A = w_success*A['success'] - w_collision*A['collisions']
    score_B = w_success*B['success'] - w_collision*B['collisions']
    print('A score', score_A, 'B score', score_B, 'A preferred?', score_A > score_B)

## 3.1 Code reading guide

When you read the previous cell, do not treat it as a black box. Trace it in this order:

1. **Inputs:** what are the given numbers, observations, states, rewards, or measurements?
2. **Internal variables:** what does each variable represent physically or mathematically?
3. **Update rule:** which line is the core mechanism from the math section?
4. **Output:** what should change if the mechanism is working?
5. **Failure case:** what parameter could make the example unstable, wrong, or unsafe?

This habit is the bridge between toy examples and real robotics code: every simulator, ROS node, policy, controller, or perception model still has inputs, state, an update rule, and outputs.

## 4. Framework/practice view

Frameworks include TRL for language-model RLHF and preference-optimization methods. Robotics analogs include human rankings of trajectories and demonstrations.

The goal is not to replace understanding with APIs. The goal is to recognize the same mechanism when a library hides the details.

In [ ]:
print('Framework keywords: Hugging Face TRL, DPO, reward models, preference datasets, human-in-the-loop robotics.')

## 4.1 Framework comparison checklist

After running or reading the framework cell, write a small mapping table for yourself:

| Question | Your answer |
|---|---|
| What object/function in the framework replaces the scratch code? |  |
| Which parameters match the math symbols? |  |
| What details does the framework hide? |  |
| What new engineering concerns appear? | installation, devices, logging, data formats, batching, safety, versioning |

This is where top-down learning becomes useful: you learn the professional API **without losing the mechanism**

## 5. Real-system connection

A human may prefer a smooth, safe robot trajectory over a jerky but successful one. Preference learning can capture these soft objectives.

## 6. Exercises

1. Add a smoothness feature to the score.
2. Why can reward models be hacked?
3. How would you collect preferences for drone flight?

**Notebook habit:** after each exercise, add a short note explaining what changed and why it matters in a robot/car/drone/VLA stack.